# 第4回：機械学習・深層学習で投稿を分類する

**このNotebookでできるようになること**

1. 教師あり学習：検索語をラベルにして「どの話題の投稿か」を当てる分類器を作り，精度を評価する
2. **自分でラベルを付けた少量のデータ**から分類器を作り，大量の投稿に自動でラベルを付ける（プロジェクトで最も使う道具）
3. 教師なし学習：ラベルなしで投稿をグループ分けする（クラスタリング）
4. 発展：学習済みの深層学習モデルで投稿のポジティブ／ネガティブを判定する（感情分析）

第3回で保存した `data/posts_words.csv` を使います（無ければサンプルデータを使います）．

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bsky_utils import *

pd.set_option("display.max_colwidth", 60)
set_japanese_font()

df = load_posts("data/posts_words.csv")
if "words_str" not in df.columns:                      # 第3回を飛ばした場合はここで形態素解析する
    df["words_str"] = df["text"].map(lambda s: " ".join(tokenize(s)))
df["words"] = df["words_str"].fillna("").str.split()
print(df["query"].value_counts())

## 1. 教師あり学習①：どの話題の投稿かを当てる

**教師あり学習**は「入力（本文）と正解（ラベル）」のペアから，入力→正解の対応を学ぶ方法です．
まずは，検索語 `query` を正解ラベルとみなして分類器を作ります（正解がすでにあるので，手順の練習に最適）．

手順：
1. 本文を数値ベクトルにする（**TF-IDF**：各単語の重要度を並べたベクトル）
2. データを **学習用** と **評価用** に分ける（覚えたデータで採点しないため）
3. 学習用で **ロジスティック回帰** を学習
4. 評価用で正解率を測る

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# ラベルが2種類以上あるデータを用意する（検索語が1つだけならサンプルデータで練習）
if df["query"].nunique() < 2:
    print("検索語が1つだけなので，サンプルデータで練習します．")
    df_sup = load_posts("sample_data/posts_sample.csv")
    df_sup["words_str"] = df_sup["text"].map(lambda s: " ".join(tokenize(s)))
else:
    df_sup = df.copy()

X_text = df_sup["words_str"].fillna("")
y = df_sup["query"]

X_train, X_test, y_train, y_test = train_test_split(X_text, y, test_size=0.3, random_state=0, stratify=y)
print("学習用:", len(X_train), "件  評価用:", len(X_test), "件")

vec = TfidfVectorizer(token_pattern=r"(?u)\S+")      # 空白区切りの単語をそのまま使う
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)
print("特徴（単語）の数:", len(vec.get_feature_names_out()))

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train)

pred = clf.predict(X_test_vec)
print("正解率:", round(accuracy_score(y_test, pred), 3))
print(classification_report(y_test, pred))

In [ ]:
# 混同行列：どのラベルをどのラベルと間違えたか
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred, labels=clf.classes_), display_labels=clf.classes_).plot(ax=ax, colorbar=False)
plt.title("混同行列（行：正解，列：予測）")
plt.tight_layout()
plt.show()

In [ ]:
# 分類の根拠になった単語：各ラベルで係数が大きい単語 Top10
terms = np.array(vec.get_feature_names_out())
coef = clf.coef_ if len(clf.classes_) > 2 else np.vstack([-clf.coef_[0], clf.coef_[0]])
for label, row in zip(clf.classes_, coef):
    idx = row.argsort()[::-1][:10]
    print(f"[{label}] " + ", ".join(terms[idx]))

> 検索語そのものが本文に入っているので当てやすいのは当然です．重要なのは **「学習→評価→根拠の確認」という手順** と，評価用データを分ける理由を理解することです．

## 2. 教師あり学習②：自分でラベルを付けて分類する（プロジェクト用の道具）

プロジェクトで本当に知りたいのは，「防災の投稿のうち **要望** はどれくらいか」「**苦情**と**称賛**の比率は」のような，データに最初から付いていない分類です．

そのときは次の手順を使います．

1. 投稿を100〜200件ほど取り出して **人手でラベルを付ける**（Excelで `label` 列に記入）
2. それを学習データにして分類器を作る
3. 残りの数千件に **自動でラベルを付ける**
4. 精度（評価用データでの正解率）と一緒に結果を報告する

### 2-1. ラベル付け用シートを書き出す

`data/labeling_sheet.csv` をExcelで開き，`label` 列に分類名を入力して保存します（例：`情報提供` / `意見・要望` / `体験・感想` / `宣伝・告知`）．

In [ ]:
sheet = df.sample(n=min(150, len(df)), random_state=0)[["post_id", "query", "text"]].copy()
sheet["label"] = ""
sheet.to_csv("data/labeling_sheet.csv", index=False, encoding="utf-8-sig")
print("data/labeling_sheet.csv を書き出しました（", len(sheet), "件）．Excelで label 列を記入してください．")
sheet.head()

### 2-2. ラベル付きデータで学習する

自分たちのラベルがまだ無い場合は，授業用のラベル付きサンプル（`sample_data/labeled_sample.csv`，合成データ）で手順を確認します．

In [ ]:
own = pd.read_csv("data/labeling_sheet.csv", encoding="utf-8-sig")
own = own[own["label"].fillna("").astype(str).str.strip() != ""]

if len(own) >= 40 and own["label"].nunique() >= 2:
    labeled = own
    print("自分たちのラベル付きデータを使います:", len(labeled), "件")
else:
    labeled = pd.read_csv("sample_data/labeled_sample.csv", encoding="utf-8-sig")
    print("ラベルがまだ無いので，サンプルのラベル付きデータで練習します:", len(labeled), "件")

labeled["words_str"] = labeled["text"].map(lambda s: " ".join(tokenize(s)))
print(labeled["label"].value_counts())

In [ ]:
from sklearn.pipeline import make_pipeline

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    labeled["words_str"], labeled["label"], test_size=0.3, random_state=0, stratify=labeled["label"])

# ベクトル化と分類器を1つにまとめた「パイプライン」
model = make_pipeline(TfidfVectorizer(token_pattern=r"(?u)\S+"), LogisticRegression(max_iter=1000))
model.fit(Xl_train, yl_train)

pred_l = model.predict(Xl_test)
print("正解率:", round(accuracy_score(yl_test, pred_l), 3))
print(classification_report(yl_test, pred_l))

### 2-3. 全投稿に自動でラベルを付ける

In [ ]:
df["pred_label"] = model.predict(df["words_str"].fillna(""))
df["pred_prob"] = model.predict_proba(df["words_str"].fillna("")).max(axis=1).round(2)   # 予測の確信度

dist = df.groupby(["query", "pred_label"]).size().unstack(fill_value=0)
dist.plot(kind="bar", stacked=True, figsize=(7, 4))
plt.ylabel("投稿数")
plt.title("自動ラベルの分布（検索語別）")
plt.tight_layout()
plt.show()

df[["query", "text", "pred_label", "pred_prob"]].sort_values("pred_prob", ascending=False).head(10)

> **報告のしかた**：「分類器の正解率は評価用データで 0.85 だった．この分類器を全 2,300 件に適用したところ，要望は 18% だった」のように，**精度と一緒に**結果を示します．確信度（`pred_prob`）が低い投稿を目視で確認するのも有効です．

## 3. 教師なし学習：ラベルなしで投稿をグループ分けする（クラスタリング）

ラベルを付ける前に「そもそもどんな種類の投稿があるのか」を知りたいときは，**クラスタリング**が役立ちます．
ここでは K-means で投稿を `K` 個のグループに分け，各グループの特徴語を見ます．

In [ ]:
from sklearn.cluster import KMeans

K = 5                                                   # ← グループ数．3〜8くらいで変えてみる
vec_all = TfidfVectorizer(token_pattern=r"(?u)\S+", min_df=2)
X_all = vec_all.fit_transform(df["words_str"].fillna(""))

km = KMeans(n_clusters=K, random_state=0, n_init=10)
df["cluster"] = km.fit_predict(X_all)

terms_all = np.array(vec_all.get_feature_names_out())
for k in range(K):
    idx = km.cluster_centers_[k].argsort()[::-1][:8]
    n = int((df["cluster"] == k).sum())
    print(f"クラスタ{k}（{n}件）: " + ", ".join(terms_all[idx]))

In [ ]:
# 各クラスタの投稿例を見る（特徴語だけでは分からない中身を確認する）
for k in range(K):
    print(f"\n=== クラスタ{k} ===")
    for text in df[df["cluster"] == k]["text"].head(3):
        print(" -", text[:70])

In [ ]:
# クラスタと検索語の関係
pd.crosstab(df["cluster"], df["query"])

> クラスタは「機械が見つけた似たもの同士のまとまり」であって，意味のある分類とは限りません．**人が中身を読んで名前を付ける**ところまでが分析です．

## 4. 発展：感情分析（ポジティブ／ネガティブの判定）

ここまでの分類器は，自分たちで付けたラベルを学習しました．一方，「投稿がポジティブかネガティブか」のような **汎用的な分類** は，すでに大量のデータで学習済みの **深層学習モデル（BERT 系）** を Hugging Face から取り寄せて使えます．学習は不要で，推論だけなら **CPU でも動きます**（モデルのダウンロードに数百MB，1,000件の判定に数分）．

| 用途 | モデル名（Hugging Face） | ラベル |
|---|---|---|
| 日本語 | `koheiduck/bert-japanese-finetuned-sentiment` | NEGATIVE / NEUTRAL / POSITIVE |
| 日本語・英語など多言語（SNS向け） | `cardiffnlp/twitter-xlm-roberta-base-sentiment` | negative / neutral / positive |
| 英語（SNS向け） | `cardiffnlp/twitter-roberta-base-sentiment-latest` | negative / neutral / positive |

### 事前準備

`pbl2026` 環境に追加のライブラリを入れます（初回のみ．合計で 500MB ほど）．Google Colab で行う場合は先頭に `!` を付けて同じコマンドを実行します．

```bash
conda activate pbl2026
pip install transformers torch fugashi unidic-lite sentencepiece
```

> 単純な「ポジティブ語／ネガティブ語を数える」方法は，否定（「悪くない」），皮肉，文脈に弱いので本教材では使いません．学習済みモデルでも誤りはあるため，**必ず一部を人手で確認して精度を測ってから**，全体の集計を発表に使ってください．

In [ ]:
# [SKIP-TEST]  ※ 事前に pip install transformers torch fugashi unidic-lite sentencepiece
from transformers import pipeline

clf_ja = pipeline("sentiment-analysis", model="koheiduck/bert-japanese-finetuned-sentiment")   # 初回はダウンロードに時間がかかる

examples = [
    "非常食のアルファ米を試食してみたら意外とおいしかった。",
    "防災無線が全然聞こえない。雨の音で何を言っているのか分からない。",
    "避難所が開設されました。場所は市民体育館です。",
]
for text, result in zip(examples, clf_ja(examples)):
    print(f"{result['label']:9s} {result['score']:.2f}  {text}")

### 投稿データに適用する

まず 100 件程度で試し，結果を目で確認してから全体に広げます．長い本文は先頭 200 文字に切ります（モデルの入力長の上限のため）．

In [ ]:
# [SKIP-TEST]
sample = df.sample(n=min(100, len(df)), random_state=0).copy()
results = clf_ja(sample["text"].str[:200].tolist(), batch_size=16, truncation=True)
sample["sentiment"] = [r["label"] for r in results]
sample["sentiment_score"] = [round(r["score"], 2) for r in results]

print(sample.groupby(["query", "sentiment"]).size().unstack(fill_value=0))
sample[["query", "text", "sentiment", "sentiment_score"]].sort_values("sentiment_score", ascending=False).head(10)

In [ ]:
# [SKIP-TEST]
# 検索語ごとの割合（%）を積み上げ棒グラフにする
dist = sample.groupby(["query", "sentiment"]).size().unstack(fill_value=0)
(dist.div(dist.sum(axis=1), axis=0) * 100).plot(kind="bar", stacked=True, figsize=(6, 4), colormap="RdYlGn", rot=0)
plt.ylabel("割合（%）")
plt.title("感情分析の結果（検索語別）")
plt.legend(title="", loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()
plt.show()

### 日本語と英語を同じモデルで判定する（多言語モデル）

英語圏の投稿（`04_english_analysis.ipynb`）と比べるときは，日英どちらにも使える多言語モデルを使うと，同じ基準で比較できます．

In [ ]:
# [SKIP-TEST]
clf_multi = pipeline("sentiment-analysis", model="cardiffnlp/twitter-xlm-roberta-base-sentiment")

texts = [
    "避難所の案内が分かりにくい。看板をもっと増やしてほしい。",
    "The evacuation center staff were incredibly helpful and kind.",
    "Polls open tomorrow in the national election.",
]
for text, result in zip(texts, clf_multi(texts)):
    print(f"{result['label']:9s} {result['score']:.2f}  {text}")

### 精度を確かめる（発表で数字を使う前に必ず行う）

1. `data/labeling_sheet.csv` と同じ要領で 50〜100 件を取り出し，人手で `positive / neutral / negative` を付ける．
2. モデルの判定と比べて正解率を出す（`accuracy_score`）．
3. 正解率と「NEUTRAL が多い」「皮肉に弱い」などの傾向を，結果と一緒に報告する．

```python
from sklearn.metrics import accuracy_score, classification_report
human = pd.read_csv("data/sentiment_check.csv", encoding="utf-8-sig")        # 列: post_id, text, label（人手）
pred = [r["label"].lower() for r in clf_ja(human["text"].str[:200].tolist(), truncation=True)]
print("正解率:", accuracy_score(human["label"].str.lower(), pred))
print(classification_report(human["label"].str.lower(), pred))
```

### Google Colab で実行する場合

1. https://colab.research.google.com/ で新しいノートブックを作り，ランタイム → ランタイムのタイプを変更 → **GPU（T4）** を選ぶ（CPU でも動くが遅い）．
2. 次を実行してライブラリを入れる．
   ```
   !pip install transformers torch fugashi unidic-lite sentencepiece
   ```
3. 左のファイル欄に `data/posts.csv` をアップロードし，上のセルと同じコードを実行する．
4. 結果の CSV（`sample.to_csv("sentiment_result.csv", index=False, encoding="utf-8-sig")`）をダウンロードして手元の Notebook で集計・可視化する．

### 英語圏の投稿を分析する

世界情勢やグローバルな話題は英語の投稿のほうが圧倒的に多く，Bluesky のトレンドも英語圏中心です．英語での収集・前処理・比較の方法は [`04_english_analysis.ipynb`](04_english_analysis.ipynb) にまとめています．

## まとめ：3回の演習で手に入れた道具

| 段階 | 道具 |
|---|---|
| 第2回 データ取得 | キーワード・期間・アカウント・返信・画像の取得，CSV保存と取得条件の記録 |
| 第3回 加工・可視化 | 形態素解析，頻出語，ワードクラウド，時系列，共起ネットワーク，TF-IDF |
| 第4回 機械学習 | 教師あり分類（自分のラベルで学習→大量データに適用），クラスタリング，感情分析（学習済みモデル） |

**プロジェクトの考え方**：「社会の誰の，どんな困りごとを，SNSのデータで見える化・改善できるか」を先に決め，その問いに答えるために上の道具を組み合わせてください．道具を全部使う必要はありません．